# Pronóstico de caudal – Cuenca del Gualeguaychú

Calcula, a partir de un **CSV de precipitación horaria pronosticada** (grilla lat/lon) y de la capa de **subcuencas HydroSHEDS nivel 12**:

1. la **lámina de lluvia horaria (mm) de cada subcuenca** (interpolación IDW),
2. los **eventos de lluvia** y la **humedad antecedente (AMC)** de cada uno,
3. la lluvia efectiva (SCS-CN, **por evento**) y el hidrograma de cada subcuenca con su **Tc**,
4. el **tránsito por la red** siguiendo `NEXT_DOWN`, tramo por tramo, con **Muskingum** (o retardo puro),
5. el hidrograma en el **punto más bajo de descarga** (+ caudal base), con una **banda de incertidumbre** según el CN.

**Cómo usarlo:** ejecutá las celdas en orden (*Entorno → Ejecutar todo*). Ajustá los parámetros en la celda 2 y subí los archivos cuando se pida. Al final se descarga un `.zip` con el Excel, las figuras y la capa para QGIS.

> El Tc se estima con fórmula (Kirpich/Témez) si no se cargan valores propios. Para usar los propios: corré una vez, editá `Tc_h` y `CN` en la hoja `Parametros_subcuencas` del Excel y volvé a cargarlo marcando *usar_parametros*.

**Cambios respecto de la versión anterior**
- El SCS-CN se aplica **por evento** (separados por `horas_sin_lluvia` horas secas). Antes se acumulaba toda la semana y las lloviznas de los primeros días "gastaban" la abstracción inicial, lo que sobrestimaba el pico.
- **AMC**: con `amc = "Auto"` y la lluvia observada de los 5 días previos (`p5_previa`), el CN se corrige a condición seca (I) o húmeda (III) en cada evento. Con `"II"` se comporta como antes.
- **Tránsito Muskingum por la red**: cada subcuenca descarga en la de abajo (`NEXT_DOWN`) y la onda se retrasa y atenúa en cada tramo (K = largo / velocidad). El largo sale de la **diferencia** de `DIST_SINK`, que mide hasta el mar y no hasta la salida del Gualeguaychú.
- **Ensamble de CN** (`ensamble_dcn`) para ver cuánto cambia el pico si el CN está mal estimado, y **balance de masa** como control.
- Opción para CSV en **UTC**. Nuevas hojas en el Excel: `Eventos`, `Q_salida_ensamble_CN`, `Q_local_m3s`, `Q_acumulado_m3s`.

In [ ]:
# 1) Librerías (Colab ya trae casi todo; geopandas se instala si falta)
try:
    import geopandas as gpd
except ImportError:
    !pip install -q geopandas
    import geopandas as gpd
!pip install -q openpyxl

import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
# 2) Parámetros del modelo
cn_def = 95            #@param {type:"number"}
metodo_tc = "Kirpich"  #@param ["Kirpich", "Temez"]
pendiente = 0.002      #@param {type:"number"}
velocidad = 1.0        #@param {type:"number"}
q_base = 5.0           #@param {type:"number"}
ia_ratio = 0.2         #@param {type:"number"}
usar_parametros = False #@param {type:"boolean"}

# --- Lluvia efectiva por eventos y humedad antecedente (AMC)
horas_sin_lluvia = 24  #@param {type:"integer"}
amc = "II"             #@param ["II", "Auto", "I", "III"]
p5_previa = 0.0        #@param {type:"number"}
epoca = "Crecimiento"  #@param ["Crecimiento", "Latencia"]

# --- Tránsito por la red de subcuencas
transito = "Muskingum"  #@param ["Muskingum", "Retardo puro"]
x_muskingum = 0.2       #@param {type:"number"}

# --- Incertidumbre: corrimientos de CN a simular (el 0 siempre se incluye)
ensamble_dcn = "-10, -5, 0, 5"  #@param {type:"string"}

# --- Hora en la que viene el CSV de pronóstico
huso_csv = "Hora local (UTC-3)"  #@param ["Hora local (UTC-3)", "UTC"]

# cn_def          : CN por defecto (AMC II) para subcuencas sin CN propio
# metodo_tc       : fórmula de Tc si no se carga Tc propio (L = 1.4*A^0.6 km)
# pendiente       : pendiente media del cauce [m/m], para la fórmula de Tc
# velocidad       : velocidad de la onda en el cauce [m/s]; define K = largo del tramo / velocidad
# q_base          : caudal base [m3/s]
# ia_ratio        : abstracción inicial Ia = ia_ratio * S
# horas_sin_lluvia: horas secas seguidas que separan dos eventos. El SCS-CN se aplica a cada
#                   evento por separado (el suelo "se recupera" entre tormentas)
# amc             : humedad antecedente. "II" = normal (como antes). "Auto" la elige en cada
#                   evento según la lluvia de los 5 días previos (pronóstico + p5_previa)
# p5_previa       : lluvia OBSERVADA en los 5 días anteriores al pronóstico [mm] (para "Auto")
# epoca           : época del año para los umbrales de AMC (Crecimiento = primavera-verano)
# transito        : "Muskingum" atenúa y retrasa la onda tramo por tramo siguiendo NEXT_DOWN;
#                   "Retardo puro" solo la desplaza en el tiempo (comportamiento anterior)
# x_muskingum     : factor de ponderación X de Muskingum (0 = máxima atenuación, 0.25 = poca)
# ensamble_dcn    : se corre el modelo con CN + cada valor, para ver la banda de incertidumbre
# huso_csv        : si el CSV viene en UTC, se pasa a hora argentina (UTC-3)

In [ ]:
# 3) Funciones del modelo
def leer_csv_pronostico(archivos):
    """Devuelve (lon, lat, matriz[npuntos, ntiempos], tiempos)."""
    partes = []
    for archivo in sorted(archivos):
        d = pd.read_csv(archivo, encoding="utf-8-sig")
        d.columns = [c.strip().lower() for c in d.columns]
        falt = {"latitud", "longitud", "fecha_hora_pronostico", "precipitacion_mm"} - set(d.columns)
        if falt:
            raise ValueError(f"{os.path.basename(archivo)}: faltan columnas {sorted(falt)}")
        d["fecha_hora_pronostico"] = pd.to_datetime(d["fecha_hora_pronostico"])
        partes.append(d[["latitud", "longitud", "fecha_hora_pronostico", "precipitacion_mm"]])
    d = pd.concat(partes, ignore_index=True)
    d["precipitacion_mm"] = pd.to_numeric(d["precipitacion_mm"], errors="coerce").clip(lower=0)
    d = d.drop_duplicates(["latitud", "longitud", "fecha_hora_pronostico"], keep="last")
    tabla = d.pivot_table(index=["latitud", "longitud"], columns="fecha_hora_pronostico",
                          values="precipitacion_mm", aggfunc="mean").sort_index(axis=1)
    lat = tabla.index.get_level_values(0).values
    lon = tabla.index.get_level_values(1).values
    return lon, lat, tabla.values, tabla.columns


def leer_capa(ruta):
    """Lee shp / gpkg / geojson, o un .zip con el shapefile."""
    if ruta.lower().endswith(".zip"):
        return gpd.read_file("zip://" + ruta)
    return gpd.read_file(ruta)


def leer_parametros(ruta):
    """Archivo opcional con HYBAS_ID (o ID), Tc_h y/o CN. Devuelve DataFrame indexado por ID (str)."""
    if ruta.lower().endswith((".xlsx", ".xls")):
        hojas = pd.ExcelFile(ruta).sheet_names
        hoja = "Parametros_subcuencas" if "Parametros_subcuencas" in hojas else hojas[0]
        d = pd.read_excel(ruta, sheet_name=hoja, dtype=str)
    else:
        d = pd.read_csv(ruta, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
    d.columns = [c.strip().lower() for c in d.columns]
    c_id = next((c for c in ("hybas_id", "id") if c in d.columns), None)
    if c_id is None:
        raise ValueError("El archivo de parámetros necesita una columna HYBAS_ID (o ID).")
    out = pd.DataFrame(index=d[c_id].astype(str).str.strip().str.replace(r"\.0$", "", regex=True))
    for col, nombre in (("tc_h", "Tc_h"), ("cn", "CN")):
        if col in d.columns:
            out[nombre] = pd.to_numeric(d[col].str.replace(",", "."), errors="coerce").values
    return out


# --------------------------------------------------------------------------
# Lámina media por subcuenca
# --------------------------------------------------------------------------
def puntos_en_poligono(geom, paso=0.02):
    """Puntos de una grilla regular (paso en grados, ~2 km) dentro del polígono."""
    minx, miny, maxx, maxy = geom.bounds
    xs = np.arange(minx + paso / 2, maxx, paso)
    ys = np.arange(miny + paso / 2, maxy, paso)
    X, Y = np.meshgrid(xs, ys)
    X, Y = X.ravel(), Y.ravel()
    try:
        import shapely
        dentro = shapely.contains_xy(geom, X, Y)
    except AttributeError:  # shapely 1.x
        from shapely import vectorized
        dentro = vectorized.contains(geom, X, Y)
    pts = np.column_stack([X[dentro], Y[dentro]])
    if len(pts) == 0:  # subcuenca más chica que la malla
        p = geom.representative_point()
        pts = np.array([[p.x, p.y]])
    return pts


def interpolar_idw(P, V, S, k=4, pot=2.0):
    """Inverso de la distancia con los k vecinos más cercanos (solo numpy).
    P[n,2]=(lon,lat) de los puntos, V[n,nt] valores, S[m,2] puntos a estimar."""
    f = np.cos(np.deg2rad(P[:, 1].mean()))  # corrige la longitud por la latitud
    dx = (S[:, None, 0] - P[None, :, 0]) * f
    dy = S[:, None, 1] - P[None, :, 1]
    d2 = dx * dx + dy * dy
    k = min(k, P.shape[0])
    idx = np.argpartition(d2, k - 1, axis=1)[:, :k]
    dk = np.take_along_axis(d2, idx, axis=1)
    w = 1.0 / np.maximum(dk, 1e-12) ** (pot / 2.0)
    w /= w.sum(axis=1, keepdims=True)
    return np.einsum("mk,mkt->mt", w, V[idx])


def lamina_por_subcuenca(gdf_wgs84, lon, lat, matriz):
    """Lámina media (mm) por hora y subcuenca. Array [ntiempos, nsubcuencas]."""
    P = np.column_stack([lon, lat])
    V = np.nan_to_num(matriz, nan=0.0)
    out = np.zeros((V.shape[1], len(gdf_wgs84)))
    for k, geom in enumerate(gdf_wgs84.geometry):
        S = puntos_en_poligono(geom)
        out[:, k] = np.clip(interpolar_idw(P, V, S), 0, None).mean(axis=0)
    return out


# --------------------------------------------------------------------------
# Parámetros de subcuencas y red de drenaje
# --------------------------------------------------------------------------
def buscar_columna(gdf, candidatos):
    cols = {c.lower(): c for c in gdf.columns}
    for c in candidatos:
        if c.lower() in cols:
            return cols[c.lower()]
    return None


def como_id(serie):
    """IDs como texto, sin el '.0' que aparece si la columna se leyó como número decimal."""
    return serie.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


def tc_formula(area_km2, metodo, pend):
    """Tc (h) con L = 1.4*A^0.6 km (Hack) y pendiente media del cauce (m/m)."""
    L = 1.4 * np.power(area_km2, 0.6)  # km
    if metodo == "Temez":
        return 0.3 * (L / pend ** 0.25) ** 0.76
    return 0.0195 * (L * 1000.0) ** 0.77 * pend ** -0.385 / 60.0  # Kirpich


def orden_topologico(ids, down):
    """Ordena las subcuencas de aguas arriba hacia aguas abajo (algoritmo de Kahn)."""
    n_arriba = {i: 0 for i in ids}
    for d in down:
        if d in n_arriba:
            n_arriba[d] += 1
    abajo = dict(zip(ids, down))
    pila = [i for i in ids if n_arriba[i] == 0]
    orden = []
    while pila:
        i = pila.pop()
        orden.append(i)
        d = abajo[i]
        if d in n_arriba:
            n_arriba[d] -= 1
            if n_arriba[d] == 0:
                pila.append(d)
    if len(orden) != len(ids):
        raise ValueError("NEXT_DOWN forma un ciclo: revisá la capa de subcuencas.")
    return orden


def preparar_subcuencas(gdf, cn_def, vel_ms, metodo_tc, pend, df_param=None):
    """Una fila por subcuenca: ID, Area_km2, Tc_h, CN, Down_ID (a quién drena), Tramo_km y K_h
    (largo y tiempo de viaje hasta la salida de la subcuenca de abajo), Tv_h (tiempo de viaje
    hasta la salida de la cuenca), Es_salida y Llega_a_salida. Devuelve (subc, gdf, orden)."""
    gdf = gdf.reset_index(drop=True).copy()
    gm = gdf.to_crs(epsg=6933)  # áreas iguales, en metros
    cen = gm.geometry.centroid

    c_id = buscar_columna(gdf, ["HYBAS_ID", "ID"])
    ids = como_id(gdf[c_id]) if c_id else pd.Series(gdf.index.astype(str))

    c_a = buscar_columna(gdf, ["SUB_AREA", "Area_km2"])
    area = gdf[c_a].astype(float) if c_a else gm.geometry.area / 1e6

    # --- Tc y CN
    tc = pd.Series(tc_formula(area.values, metodo_tc, pend), index=gdf.index)
    c_tc = buscar_columna(gdf, ["Tc_h"])
    if c_tc:
        tc = gdf[c_tc].astype(float).fillna(tc)
    cn = pd.Series(cn_def, index=gdf.index, dtype=float)
    c_cn = buscar_columna(gdf, ["CN"])
    if c_cn:
        cn = gdf[c_cn].astype(float).fillna(cn)
    if df_param is not None:
        if "Tc_h" in df_param:
            tc = ids.map(df_param["Tc_h"]).astype(float).fillna(tc)
        if "CN" in df_param:
            cn = ids.map(df_param["CN"]).astype(float).fillna(cn)

    # --- Red de drenaje: a qué subcuenca le tira el agua cada una
    c_nd = buscar_columna(gdf, ["NEXT_DOWN"])
    c_ds = buscar_columna(gdf, ["DIST_SINK", "DIST_MAIN"])
    c_up = buscar_columna(gdf, ["UP_AREA"])
    if c_nd:
        down = como_id(gdf[c_nd])
        down = down.where(down.isin(set(ids)), "")  # "" = drena fuera de la capa
        ref = gdf[c_up].astype(float) if c_up else area
        i_sal = ref[down == ""].idxmax()  # varias salidas: la de mayor área aportante
    else:  # sin topología: todas drenan directo a la más al sur
        i_sal = cen.y.idxmin()
        down = pd.Series(ids[i_sal], index=gdf.index)
        down[i_sal] = ""

    # --- Largo de cada tramo (de la salida de una subcuenca a la salida de la de abajo).
    # Se usa la DIFERENCIA de DIST_SINK: DIST_SINK mide hasta el mar, no hasta nuestra salida.
    pos = {k: i for i, k in enumerate(ids)}
    ds = gdf[c_ds].astype(float).values if c_ds else None
    tramo = np.zeros(len(gdf))
    for i, d in enumerate(down):
        if d == "":
            continue
        k = pos[d]
        if ds is not None and ds[i] - ds[k] > 0:
            tramo[i] = ds[i] - ds[k]
        else:  # sin DIST_SINK: distancia entre centroides con un factor de sinuosidad 1.3
            tramo[i] = cen.iloc[i].distance(cen.iloc[k]) * 1.3 / 1000.0
    K = tramo * 1000.0 / vel_ms / 3600.0

    subc = pd.DataFrame({"ID": ids.values, "Area_km2": area.values, "Tc_h": tc.values,
                         "CN": cn.values, "Down_ID": down.values, "Tramo_km": tramo, "K_h": K})
    orden = orden_topologico(list(subc["ID"]), list(subc["Down_ID"]))

    # Tiempo de viaje hasta la salida y si el agua llega a la salida principal
    sal = ids[i_sal]
    abajo, Kd = dict(zip(subc["ID"], subc["Down_ID"])), dict(zip(subc["ID"], subc["K_h"]))
    tv, llega = {}, {}
    for i in reversed(orden):  # de aguas abajo hacia aguas arriba
        d = abajo[i]
        tv[i] = 0.0 if d == "" else Kd[i] + tv[d]
        llega[i] = (i == sal) if d == "" else llega[d]
    subc["Tv_h"] = subc["ID"].map(tv)
    subc["Es_salida"] = subc["ID"] == sal
    subc["Llega_a_salida"] = subc["ID"].map(llega)
    if c_up:
        subc["Area_aportante_km2"] = gdf[c_up].values
    return subc, gdf, orden


# --------------------------------------------------------------------------
# Lluvia efectiva: SCS-CN por eventos con humedad antecedente
# --------------------------------------------------------------------------
def lluvia_efectiva_scs(p_inc, cn, ia_ratio=0.2):
    """Lluvia efectiva incremental (mm) con SCS-CN sobre la precipitación acumulada."""
    s = 25400.0 / cn - 254.0
    ia = ia_ratio * s
    pc = np.cumsum(p_inc)
    pec = np.where(pc > ia, (pc - ia) ** 2 / (pc - ia + s), 0.0)
    return np.diff(np.concatenate([[0.0], pec]))


def cn_amc(cn2, clase):
    """Pasa un CN de condición II (normal) a condición I (seco) o III (húmedo)."""
    if clase == "I":
        return cn2 / (2.281 - 0.01281 * cn2)
    if clase == "III":
        return cn2 / (0.427 + 0.00573 * cn2)
    return cn2


def clase_amc(p5, epoca):
    """Clase AMC (SCS) según la lluvia de los 5 días previos, en mm."""
    lim = (12.7, 27.9) if epoca == "Latencia" else (35.6, 53.3)
    return "I" if p5 < lim[0] else ("III" if p5 > lim[1] else "II")


def detectar_eventos(lluvia, horas_sin_lluvia=24, umbral=0.1):
    """Agrupa las horas con lluvia (> umbral mm/h en alguna subcuenca) en eventos separados por
    al menos `horas_sin_lluvia` horas secas. Devuelve [(i_inicio, i_fin), ...] (inclusive)."""
    idx = np.flatnonzero((lluvia.values > umbral).any(axis=1))
    if len(idx) == 0:
        return []
    eventos, i0, prev = [], idx[0], idx[0]
    for i in idx[1:]:
        if i - prev - 1 >= horas_sin_lluvia:
            eventos.append((i0, prev))
            i0 = i
        prev = i
    eventos.append((i0, prev))
    return eventos


def tabla_eventos(lluvia_media, eventos, amc, p5_previa, epoca):
    """Una fila por evento con su lluvia, la lluvia de los 5 días previos y la clase AMC."""
    v, filas = lluvia_media.values, []
    for k, (i0, i1) in enumerate(eventos):
        # 5 días previos = lo que ya llovió en el pronóstico + la parte de p5_previa que cae
        # dentro de la ventana (se reparte uniforme en las 120 h anteriores al pronóstico)
        p5 = v[max(0, i0 - 120):i0].sum() + p5_previa * max(0, 120 - i0) / 120.0
        clase = amc if amc in ("I", "II", "III") else clase_amc(p5, epoca)
        filas.append({"Evento": k + 1, "Inicio": lluvia_media.index[i0],
                      "Fin": lluvia_media.index[i1], "Horas": i1 - i0 + 1,
                      "Lluvia_media_mm": round(float(v[i0:i1 + 1].sum()), 2),
                      "P5_antecedente_mm": round(float(p5), 1), "AMC": clase})
    cols = ["Evento", "Inicio", "Fin", "Horas", "Lluvia_media_mm", "P5_antecedente_mm", "AMC"]
    return pd.DataFrame(filas, columns=cols)


# --------------------------------------------------------------------------
# Hidrograma unitario y tránsito
# --------------------------------------------------------------------------
def hu_triangular_scs(area_km2, tc_h, dt=1.0):
    """HU (m3/s por mm de lluvia efectiva en dt horas), paso horario."""
    tp = dt / 2.0 + 0.6 * tc_h
    tb = 2.67 * tp
    n = int(np.ceil(tb / dt)) + 1
    t = (np.arange(n) + 1) * dt
    u = np.where(t <= tp, t / tp, np.maximum((tb - t) / (tb - tp), 0.0))
    vol_m3 = area_km2 * 1e3  # 1 mm sobre el área
    return u * vol_m3 / (3600.0 * dt * u.sum())


def transitar(q, K, metodo="Muskingum", X=0.2, dt=1.0):
    """Lleva un hidrograma por un tramo con tiempo de viaje K (h).
    'Retardo puro' lo desplaza K horas sin deformarlo. 'Muskingum' además lo atenúa: divide el
    tramo en subtramos de K <= dt y usa subpasos de tiempo para que los coeficientes sean >= 0."""
    q = np.asarray(q, dtype=float)
    if K <= 1e-6:
        return q.copy()
    t = np.arange(len(q)) * dt
    if metodo != "Muskingum":
        return np.interp(t - K, t, q, left=0.0)
    X = min(max(X, 0.0), 0.25)
    n = max(1, int(np.ceil(K / dt)))      # subtramos
    Ks = K / n
    m = max(1, int(round(dt / Ks)))       # subpasos por paso horario
    ds = dt / m
    D = Ks * (1 - X) + ds / 2
    c0, c1, c2 = (ds / 2 - Ks * X) / D, (ds / 2 + Ks * X) / D, (Ks * (1 - X) - ds / 2) / D
    qs = np.interp(np.arange((len(q) - 1) * m + 1) * ds, t, q)
    for _ in range(n):
        out = np.empty_like(qs)
        out[0] = qs[0]
        for j in range(1, len(qs)):
            out[j] = c0 * qs[j] + c1 * qs[j - 1] + c2 * out[j - 1]
        qs = out
    return qs[::m]


def simular(lluvia, subc, orden, eventos, tabla_ev, P, dcn=0.0):
    """Corre el modelo con CN + dcn. Devuelve (pe, q_local, q_acum):
    q_local = hidrograma propio de cada subcuenca en su salida;
    q_acum  = caudal total que pasa por la salida de cada subcuenca (propio + lo de arriba)."""
    nT = len(lluvia)
    cola = int(np.ceil(1.5 * subc["Tv_h"].max() + 2.67 * (0.5 + 0.6 * subc["Tc_h"].max()))) + 24
    idx = pd.date_range(lluvia.index[0], periods=nT + cola, freq="h")
    pe_df = pd.DataFrame(0.0, index=lluvia.index, columns=subc["ID"])
    q_loc = pd.DataFrame(0.0, index=idx, columns=subc["ID"])
    for _, r in subc.iterrows():
        p = lluvia[r["ID"]].values
        cn2 = float(np.clip(r["CN"] + dcn, 30, 100))
        pe = np.zeros(nT)
        for (i0, i1), clase in zip(eventos, tabla_ev["AMC"]):
            cn = float(np.clip(cn_amc(cn2, clase), 30, 100))
            pe[i0:i1 + 1] = lluvia_efectiva_scs(p[i0:i1 + 1], cn, P["ia_ratio"])
        pe_df[r["ID"]] = pe
        q = np.convolve(pe, hu_triangular_scs(r["Area_km2"], r["Tc_h"]))[:len(idx)]
        q_loc.iloc[:len(q), q_loc.columns.get_loc(r["ID"])] = q

    q_acum = q_loc.copy()
    abajo, K = dict(zip(subc["ID"], subc["Down_ID"])), dict(zip(subc["ID"], subc["K_h"]))
    for i in orden:  # de aguas arriba hacia aguas abajo
        d = abajo[i]
        if d:
            q_acum[d] += transitar(q_acum[i].values, K[i], P["transito"], P["x_muskingum"])
    return pe_df, q_loc, q_acum


def leer_ensamble(texto):
    """'-10, -5, 0, 5' -> [-10.0, -5.0, 0.0, 5.0] (el 0 siempre se incluye)."""
    vals = {float(x) for x in str(texto).replace(";", ",").split(",") if x.strip()}
    return sorted(vals | {0.0})


def calcular(archivos_csv, ruta_subc, P, ruta_param=None):
    """Núcleo del cálculo. P es el diccionario de parámetros. Devuelve un diccionario."""
    df_param = leer_parametros(ruta_param) if ruta_param else None
    subc, gdf, orden = preparar_subcuencas(leer_capa(ruta_subc), P["cn_def"], P["velocidad"],
                                           P["metodo_tc"], P["pendiente"], df_param)
    gdf84 = gdf.to_crs(epsg=4326)
    salida_id = subc.loc[subc["Es_salida"], "ID"].iloc[0]

    # 1) Lámina horaria por subcuenca
    lon, lat, matriz, tiempos = leer_csv_pronostico(archivos_csv)
    tiempos = pd.DatetimeIndex(tiempos)
    if P["huso_csv"] == "UTC":
        tiempos = tiempos - pd.Timedelta(hours=3)
    lam = lamina_por_subcuenca(gdf84, lon, lat, matriz)
    lluvia = pd.DataFrame(lam, index=tiempos, columns=subc["ID"])
    full = pd.date_range(lluvia.index.min(), lluvia.index.max(), freq="h")
    faltan = len(full) - len(lluvia)
    lluvia = lluvia.reindex(full).fillna(0.0)
    lluvia.index.name = "Fecha_hora"
    pesos = subc.set_index("ID")["Area_km2"]
    lluvia_media = (lluvia * pesos).sum(axis=1) / pesos.sum()

    # 2) Eventos y humedad antecedente
    eventos = detectar_eventos(lluvia, P["horas_sin_lluvia"])
    tabla_ev = tabla_eventos(lluvia_media, eventos, P["amc"], P["p5_previa"], P["epoca"])

    # 3-5) Escorrentía, hidrogramas y tránsito, para cada CN del ensamble
    ens = {}
    for dcn in leer_ensamble(P["ensamble_dcn"]):
        pe_d, q_loc_d, q_acum_d = simular(lluvia, subc, orden, eventos, tabla_ev, P, dcn)
        ens[f"CN{dcn:+g}"] = q_acum_d[salida_id] + P["q_base"]
        if dcn == 0.0:
            pe_df, q_loc, q_acum = pe_d, q_loc_d, q_acum_d
    ens = pd.DataFrame(ens)
    q_sal = ens["CN+0"].rename("Q_salida_m3s")

    # Balance de masa: volumen de lluvia efectiva que drena a la salida vs volumen escurrido
    llega = subc.set_index("ID")["Llega_a_salida"]
    vol_pe = float((pe_df.sum() * pesos * 1e3)[llega].sum())          # m3
    vol_q = float((q_sal - P["q_base"]).sum() * 3600.0)                # m3
    area_fuera = float(pesos[~llega].sum())

    return dict(subc=subc, gdf84=gdf84, lluvia=lluvia, pe=pe_df, q_local=q_loc, q_acum=q_acum,
                q_sal=q_sal, ens=ens, eventos=tabla_ev, lluvia_media=lluvia_media,
                faltan=faltan, salida_id=salida_id, vol_pe=vol_pe, vol_q=vol_q,
                area_fuera=area_fuera)

In [ ]:
# 4) Cargar archivos
print("1) Subí la capa de subcuencas: el .zip con el shapefile (subcuencas_hydrosheds_n12.zip)")
up = files.upload()
ruta_subc = next(iter(up))

print("\n2) Subí uno o más CSV de pronóstico de precipitación")
up = files.upload()
archivos_csv = list(up.keys())

ruta_param = None
if usar_parametros:
    print("\n3) Subí el archivo de Tc/CN por subcuenca (.xlsx o .csv)")
    up = files.upload()
    ruta_param = next(iter(up))

print("\nSubcuencas:", ruta_subc, "| CSV:", archivos_csv, "| Parámetros:", ruta_param)

In [ ]:
# 5) Calcular
P = dict(cn_def=cn_def, metodo_tc=metodo_tc, pendiente=pendiente, velocidad=velocidad,
         q_base=q_base, ia_ratio=ia_ratio, horas_sin_lluvia=horas_sin_lluvia, amc=amc,
         p5_previa=p5_previa, epoca=epoca, transito=transito, x_muskingum=x_muskingum,
         ensamble_dcn=ensamble_dcn, huso_csv=huso_csv)
R = calcular(archivos_csv, ruta_subc, P, ruta_param)
subc, q_sal, lluvia_media, ens = R["subc"], R["q_sal"], R["lluvia_media"], R["ens"]
q_pico, t_pico = float(q_sal.max()), q_sal.idxmax()

print(f"Subcuenca de salida (HYBAS_ID): {R['salida_id']}")
print(f"Q pico en la salida: {q_pico:.1f} m3/s  el {t_pico:%d/%m/%Y %H:%M}  (tránsito: {transito})")
print(f"Lluvia media acumulada en la cuenca: {lluvia_media.sum():.1f} mm")
print("Q pico según el CN:  " + "   ".join(f"{c}: {ens[c].max():.0f} m3/s" for c in ens.columns))
if R["vol_pe"] > 0:
    print(f"Balance de masa: escurrido / lluvia efectiva = {R['vol_q'] / R['vol_pe']:.3f} (ideal 1.000)")
if R["area_fuera"] > 0:
    print(f"Aviso: {R['area_fuera']:.0f} km2 de subcuencas no drenan a la salida principal")
if R["faltan"]:
    print(f"Aviso: {R['faltan']} horas sin dato se rellenaron con 0")
if amc == "Auto" and p5_previa == 0:
    print("Aviso: AMC 'Auto' con p5_previa = 0 supone que los 5 días previos fueron secos")

# Eventos de lluvia detectados y humedad antecedente usada en cada uno
display(R["eventos"])

# Tabla: lámina horaria por subcuenca (mm)
R["lluvia"].round(2)

In [ ]:
# 6) Resultados: Excel, capa para QGIS y figuras
os.makedirs("resultados", exist_ok=True)
base = "resultados/pronostico_caudal_gualeguaychu"

pesos = subc.set_index("ID")["Area_km2"]
pe_med = float((R["pe"] * pesos).sum(axis=1).sum() / pesos.sum())
resumen = pd.DataFrame({
    "Parametro": ["Q pico en la salida (m3/s)", "Fecha/hora del pico",
                  "Subcuenca de salida (HYBAS_ID)", "Q base (m3/s)",
                  "Lluvia media acumulada en la cuenca (mm)",
                  "Lluvia efectiva media acumulada (mm)",
                  "Eventos de lluvia detectados", "Balance de masa (escurrido / Pe)",
                  "Horas sin dato (rellenadas con 0)", "CN por defecto (AMC II)",
                  "AMC / P5 previa (mm) / época", "Método Tc / pendiente (m/m)",
                  "Tránsito / X Muskingum", "Velocidad de la onda (m/s)", "Ia/S",
                  "Ensamble de CN", "Hora del CSV"],
    "Valor": [round(q_pico, 2), str(t_pico), R["salida_id"], q_base,
              round(float(lluvia_media.sum()), 1), round(pe_med, 2), len(R["eventos"]),
              round(R["vol_q"] / R["vol_pe"], 3) if R["vol_pe"] > 0 else "-",
              R["faltan"], cn_def, f"{amc} / {p5_previa} / {epoca}",
              f"{metodo_tc} / {pendiente}", f"{transito} / {x_muskingum}", velocidad,
              ia_ratio, ensamble_dcn, huso_csv]})

with pd.ExcelWriter(base + ".xlsx", engine="openpyxl") as xw:
    resumen.to_excel(xw, sheet_name="Resumen", index=False)
    q_sal.to_frame().join(lluvia_media.rename("Lluvia_media_mm")).to_excel(xw, sheet_name="Q_salida")
    R["ens"].to_excel(xw, sheet_name="Q_salida_ensamble_CN")
    R["eventos"].to_excel(xw, sheet_name="Eventos", index=False)
    R["lluvia"].to_excel(xw, sheet_name="Lluvia_subcuenca_mm")
    R["pe"].to_excel(xw, sheet_name="Lluvia_efectiva_mm")
    R["q_local"].to_excel(xw, sheet_name="Q_local_m3s")
    R["q_acum"].to_excel(xw, sheet_name="Q_acumulado_m3s")
    subc.rename(columns={"ID": "HYBAS_ID"}).to_excel(xw, sheet_name="Parametros_subcuencas", index=False)

# Capa con lámina acumulada y caudal pico por subcuenca
g = R["gdf84"].copy()
g["Lluvia_mm"] = R["lluvia"].sum().values
g["Pe_mm"] = R["pe"].sum().values
g["Tc_h"] = subc["Tc_h"].values
g["K_h"] = subc["K_h"].values
g["Tv_h"] = subc["Tv_h"].values
g["CN_usado"] = subc["CN"].values
g["Qpico_m3s"] = R["q_acum"].max().values  # caudal pico en la salida de cada subcuenca
g.to_file(base + "_subcuencas.gpkg", driver="GPKG")

# Figura 1: hidrograma en el punto de descarga con la banda del ensamble de CN
fig, ax = plt.subplots(figsize=(10, 5))
if ens.shape[1] > 1:
    ax.fill_between(ens.index, ens.min(axis=1), ens.max(axis=1), color="tab:blue", alpha=0.15,
                    label=f"CN {cn_def:g} {ens.columns[0][2:]} … {ens.columns[-1][2:]}")
ax.plot(q_sal.index, q_sal.values, color="tab:blue", lw=2, label=f"CN {cn_def:g}")
ax.set_ylabel("Caudal en la salida (m³/s)")
ax.set_xlabel("Fecha y hora")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.bar(lluvia_media.index, lluvia_media.values, width=1 / 24, color="tab:gray", alpha=0.6)
ax2.set_ylabel("Lluvia media en la cuenca (mm/h)")
ax2.invert_yaxis()
ax.set_title(f"Pronóstico de caudal - Pico {q_pico:.1f} m³/s ({t_pico:%d/%m %H:%M})")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(base + ".png", dpi=150)
plt.show()

# Figura 2: mapa de lámina acumulada por subcuenca
fig2, ax3 = plt.subplots(figsize=(5, 8))
g.plot(column="Lluvia_mm", cmap="Blues", legend=True, edgecolor="k", linewidth=0.4,
       ax=ax3, legend_kwds={"label": "Lluvia acumulada (mm)"})
g[subc["Es_salida"].values].plot(ax=ax3, facecolor="none", edgecolor="red", linewidth=2)
ax3.set_title("Lámina pronosticada por subcuenca (rojo: salida)")
ax3.set_xlabel("Longitud"); ax3.set_ylabel("Latitud")
fig2.tight_layout()
fig2.savefig(base + "_mapa.png", dpi=150)
plt.show()

In [ ]:
# 7) Descargar todo en un .zip
shutil.make_archive("resultados_pronostico", "zip", "resultados")
files.download("resultados_pronostico.zip")